# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fasiuu/FLYRANK_TASK1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup — shared feature builder from this repo + DuckDB connection to the hosted release.
# Token: Colab Secret HF_TOKEN (preferred) or a getpass prompt. Never paste a token into a cell.
%pip -q install duckdb
import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('FLYRANK_TASK1'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Fasiuu/FLYRANK_TASK1.git'], check=True)
sys.path.insert(0, 'FLYRANK_TASK1/work/scripts')
import numpy as np, pandas as pd
import flyrank_features as ff
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
np.random.seed(ff.SEED)
con = ff.connect()
os.makedirs('work/outputs', exist_ok=True)
print('connected; seed =', ff.SEED)

Hugging Face READ token (hf_...): ··········
connected; seed = 42


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit:** one page (`content_hash_id`) at one decision date T.

**Time windows (all derived from `fact_content_daily_performance`):**

| window | days | used for |
|---|---|---|
| feature window | (T-90, T] | every feature |
| baseline for the label | (T-60, T] | the page's and site's normal monthly level |
| outcome window | (T, T+30] | label only |

**Dates in this notebook:** T = 2026-03-31 and 2026-04-30, so the reads cover December 2025 to May 2026. June 2026 is not touched here; it is the sealed month.

**Population:** at least 200 impressions in (T-60, T], page at least 90 days old, client GSC feed starting on or before T-89. The cell below states these claims and checks them.

In [2]:
# Raw-table checks first (Parquet metadata is cheap), then the per-item panel.
fact = f"read_parquet({ff.month_paths('2025-12-01', '2026-05-31')})"
print(con.sql(f"SELECT MIN(report_date) first_day, MAX(report_date) last_day, COUNT(*) AS rows FROM {fact}").df().to_string(index=False))
cl = con.sql(f"SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start FROM read_parquet('{ff.REL}/dim_clients.parquet')").df()
for T in ff.DEV_DATES:
    ok = (pd.to_datetime(cl.gsc_data_start) <= pd.Timestamp(T) - pd.Timedelta(days=89)).sum()
    print(f'decision date {T}: {ok} of {len(cl)} clients have GSC history covering the full 90-day feature window')

# Build the per-item panel for the two development decision dates (one DuckDB scan, ~5-8 min).
import time
t0 = time.time()
panel = ff.build_panel(con, ff.DEV_DATES)
panel['decision_date'] = pd.to_datetime(panel['decision_date']).dt.strftime('%Y-%m-%d')
m = ff.model_frame(panel)
print(f'{len(panel):,} item-date rows built in {time.time()-t0:.0f}s; {len(m):,} pass the population checks')
print(panel.groupby('decision_date').agg(rows=('content_hash_id', 'size'), clients=('client_hash_id', 'nunique'),
      full_history=('full_history', 'sum')).to_string())
print(m.groupby('decision_date').agg(model_rows=('declined', 'size'), clients=('client_hash_id', 'nunique')).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 first_day   last_day     rows
2025-12-01 2026-05-31 54952339
decision date 2026-03-31: 40 of 104 clients have GSC history covering the full 90-day feature window
decision date 2026-04-30: 41 of 104 clients have GSC history covering the full 90-day feature window


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

207,726 item-date rows built in 116s; 134,757 pass the population checks
                 rows  clients  full_history
decision_date                               
2026-03-31      98357       44         84016
2026-04-30     109369       46         88972
               model_rows  clients
decision_date                     
2026-03-31          63460       25
2026-04-30          71297       27


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every field I plan to touch, sorted into exactly one bucket. Excluded fields each get a reason. The two exclusions that matter most:

- **`content_updated_date` / `last_optimized_date`.** These are export-time values (July 2026). If a page was edited in May, the field says May even when I'm pretending to stand on March 31. Using it would put future information into the features. I test what happens if I use it anyway in the leakage notebook.
- **`fact_content_query_90d`.** Its fixed window (2026-04-02 to 2026-06-30) overlaps every outcome month I use. Notebook 03 used its columns as features for a June label, which turns out to be leakage. I don't carry that forward.

In [3]:
contract = pd.DataFrame([
  ('client_hash_id / content_hash_id', 'context', '-', 'join, group, split only - pseudonyms, never features'),
  ('decision_date', 'context', '-', 'which snapshot a row belongs to'),
  ('imp_last30 / imp_prev30 / imp_prev60_90 (log)', 'feature', '(T-90, T]', 'search exposure in three 30-day blocks'),
  ('imp_momentum / imp_momentum_long', 'feature', '(T-90, T]', 'log-ratio of recent vs earlier exposure'),
  ('clicks_last30 (log) / ctr_last30', 'feature', '(T-30, T]', 'demand captured; CTR in % of impressions'),
  ('pos_last30 / pos_change / pos_volatility', 'feature', '(T-60, T]', 'impression-weighted rank, its drift, its daily spread'),
  ('days_with_imp_last30 / spike_share / weekend_share', 'feature', '(T-30, T]', 'how steady the visibility is'),
  ('content_age_days', 'feature', 'at T', 'days since creation (population: >= 90)'),
  ('word_count / search_volume (log + has_ flags)', 'feature', 'static metadata', 'depth and keyword demand; blanks flagged'),
  ('imp_next30', 'label source', '(T, T+30]', 'the outcome month - never a feature'),
  ('client_ratio / page_ratio', 'label source', '(T, T+30]', 'site-level and page-level change used to define the label'),
  ('declined', 'label', '(T, T+30]', 'page_ratio < 0.8 x client_ratio'),
  ('declined_abs', 'reference label', '(T, T+30]', 'plain >20% drop - kept only to show why it was rejected'),
  ('vanished_next30', 'context', '(T, T+30]', 'no rows at all in the outcome month - reported, not modelled'),
  ('gsc_data_start', 'context', '-', 'population filter: client feed must cover the feature window'),
  ('content_updated_date / last_optimized_date', 'excluded', 'export time', 'current values; an edit after T is future information'),
  ('fact_content_query_90d (all columns)', 'excluded', 'fixed 2026-04-02 -> 06-30', 'overlaps every label month used here'),
  ('ga4_* sessions / engagement / scroll', 'excluded', '-', 'GA4 starts Feb-Mar 2026 for most clients - too little history'),
  ('sessions_ai / ai_*', 'excluded', '-', 'too sparse to be a stable feature (lane guide)'),
  ('provider_used / model_used', 'excluded', '-', 'generation metadata; lane guide: not a model feature'),
], columns=['field', 'bucket', 'window', 'why'])
print(contract.to_string(index=False))

                                             field          bucket                    window                                                           why
                  client_hash_id / content_hash_id         context                         -          join, group, split only - pseudonyms, never features
                                     decision_date         context                         -                               which snapshot a row belongs to
     imp_last30 / imp_prev30 / imp_prev60_90 (log)         feature                 (T-90, T]                        search exposure in three 30-day blocks
                  imp_momentum / imp_momentum_long         feature                 (T-90, T]                       log-ratio of recent vs earlier exposure
                  clicks_last30 (log) / ctr_last30         feature                 (T-30, T]                      demand captured; CTR in % of impressions
          pos_last30 / pos_change / pos_volatility         feature    

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Each claim above gets a query:

1. **Grain:** zero duplicate (date, client, page) keys in the raw fact.
2. **Nulls:** impressions are never null in March 2026, but position is null on 63% of rows, because it only exists on days with impressions. The GSC/GA4 availability flags are three-valued (TRUE/FALSE/NULL), so I filter with `IS TRUE`, never `= FALSE`.
3. **Position:** I only average position over days with impressions, weighted by impressions.
4. **Windows:** the youngest page in the model frame is 90+ days old, and no feature is named after or built from the outcome window.
5. **Missingness:** `word_count` and `search_volume` gaps follow `content_type`, which is exactly the trap the dictionary warns about. So I add `has_word_count` / `has_keyword_data` flags instead of a silent `fillna(0)`.

In [4]:
one = f"read_parquet('{ff.REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
print('1) grain probe on the raw daily fact (expect 0 duplicate keys):')
print(con.sql(f"""SELECT COUNT(*) AS duplicate_keys FROM (SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
                  FROM {one} GROUP BY ALL HAVING c > 1)""").df().to_string(index=False))
print('\n2) null rates in March 2026 (the availability flags are three-valued: TRUE / FALSE / NULL):')
print(con.sql(f"""SELECT COUNT(*) AS rows,
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS null_impr,
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS null_position,
    AVG(CASE WHEN gsc_data_available IS TRUE THEN 1.0 ELSE 0 END) AS gsc_true,
    AVG(CASE WHEN gsc_data_available IS NULL THEN 1.0 ELSE 0 END) AS gsc_null,
    AVG(CASE WHEN ga4_data_available IS TRUE THEN 1.0 ELSE 0 END) AS ga4_true,
    AVG(CASE WHEN ga4_data_available IS NULL THEN 1.0 ELSE 0 END) AS ga4_null
    FROM {one}""").df().round(3).to_string(index=False))
print('\n3) position only means something on days with impressions:')
print(con.sql(f"""SELECT (gsc_impressions > 0) AS has_impr, COUNT(*) AS rows, ROUND(AVG(gsc_avg_position), 2) AS avg_pos,
    ROUND(AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END), 3) AS null_pos FROM {one} GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print('\n4) windows: no page younger than the 90-day feature window, labels only from after T:')
print('min content_age_days in the model frame:', int(m.content_age_days.min()))
print('features named after the outcome window:', [f for f in ff.FEATURES if 'next' in f] or 'none')
print('\n5) missingness before fills (panel) and the has_ flags by content type (model frame):')
print(panel[['pos_last30', 'pos_change', 'pos_volatility', 'content_age_days', 'word_count', 'search_volume']].isna().mean().round(3).to_string())
print(m.groupby('content_type').agg(rows=('declined', 'size'), has_word_count=('has_word_count', 'mean'),
      has_keyword=('has_keyword_data', 'mean')).round(3).to_string())

1) grain probe on the raw daily fact (expect 0 duplicate keys):


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 duplicate_keys
              0

2) null rates in March 2026 (the availability flags are three-valued: TRUE / FALSE / NULL):
   rows  null_impr  null_position  gsc_true  gsc_null  ga4_true  ga4_null
9841378        0.0          0.633     0.367       0.0     0.042     0.307

3) position only means something on days with impressions:
 has_impr    rows  avg_pos  null_pos
    False 6230317      NaN       1.0
     True 3611061    15.83       0.0

4) windows: no page younger than the 90-day feature window, labels only from after T:
min content_age_days in the model frame: 90
features named after the outcome window: none

5) missingness before fills (panel) and the has_ flags by content type (model frame):
pos_last30          0.013
pos_change          0.107
pos_volatility      0.013
content_age_days    0.000
word_count          0.268
search_volume       0.017
                      rows  has_word_count  has_keyword
content_type                                           
comparison article      

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

What this data can't tell me:

- **Unbalanced panel.** Clients joined at very different times. Only 40-41 of the 104 clients have a GSC feed covering the full feature window at the two snapshots, and after the other checks the model population is 25-27 clients, not 104. Results describe those clients.
- **Early GSC-only history.** GA4 starts in February/March 2026 for most clients, so engagement can't be a feature without throwing away most of the history.
- **Query table overlap.** It covers April-June 2026 only, which is the same as my outcome months, so it is excluded.
- **Edits after T.** Most panel rows belong to pages edited *after* the decision date (see `updated_after_T`). I can't see the page as it was on day T, only its metrics.
- **Site-wide swings.** The median site's impressions fell about 26% in May (median site ratio 0.74, against 0.89 in April). Part of the label is defined relative to that, which is a choice, not a fact of nature.
- **No causal reading.** Nothing here shows what a refresh would *do*; it only shows which pages tended to fall behind.

In [5]:
print('clients by GSC start quarter (unbalanced panel):')
print(pd.to_datetime(cl.gsc_data_start).dt.to_period('Q').value_counts().sort_index().to_string())
q = con.sql(f"SELECT MIN(window_start) s, MAX(window_end) e FROM read_parquet('{ff.REL}/fact_content_query_90d.parquet')").fetchone()
print(f'\nquery table window: {q[0]} -> {q[1]}  (label months used: April, May, June 2026 -> overlap)')
print(f"share of panel rows whose page was edited AFTER the decision date: {panel.updated_after_T.mean():.3f}")
print(f"share of panel rows that vanish in the outcome month: {panel.vanished_next30.mean():.3f}")
print(f"share of panel rows dropped by the population checks: {1 - len(m)/len(panel):.3f}")
print('\nsite-level swing in the outcome month (median client_ratio by snapshot):', m.groupby('decision_date').client_ratio.median().round(3).to_dict())

clients by GSC start quarter (unbalanced panel):
gsc_data_start
2025Q1     4
2025Q2     5
2025Q3    15
2025Q4    16
2026Q1    17
2026Q2    10
Freq: Q-DEC


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


query table window: 2026-04-02 -> 2026-06-30  (label months used: April, May, June 2026 -> overlap)
share of panel rows whose page was edited AFTER the decision date: 0.806
share of panel rows that vanish in the outcome month: 0.011
share of panel rows dropped by the population checks: 0.351

site-level swing in the outcome month (median client_ratio by snapshot): {'2026-03-31': 0.894, '2026-04-30': 0.737}


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.